In [14]:
%pip install faiss-cpu langchain langchain-community sentence-transformers langchain-huggingface langchain-text-splitters


   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   ---------- ----------------------------- 5.0/18.9 MB 32.0 MB/s eta 0:00:01
   ---------------------------------- ----- 16.3/18.9 MB 45.2 MB/s eta 0:00:01
   ---------------------------------------- 18.9/18.9 MB 44.8 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import os
import re
import hashlib
from pathlib import Path
import pickle

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document


In [16]:
# ──CONFIGURACIÓN GLOBAL ──────────────────────────────────────────────────
DOCS_DIR    = "RAG"          # Carpeta .txt
FAISS_DIR   = "faiss_index"  # Donde se guarda el índice persistente
CHUNK_SIZE  = 500            # Tamaño de cada chunk en caracteres
CHUNK_OVERLAP = 80           # Solapamiento para no perder contexto entre chunks
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # Modelo de embeddings

In [17]:
# ── LIMPIEZA DE TEXTO ────────────────────────────────────────────
def limpiar_texto(texto: str) -> str:
    """
    Limpia el texto crudo de un documento TXT:
    - Elimina líneas vacías múltiples
    - Normaliza espacios y saltos de línea raros
    - Quita caracteres de control
    """
    # Eliminar caracteres de control (excepto salto de línea y tabulación)
    texto = re.sub(r"[^\x09\x0A\x0D\x20-\x7E\x80-\xFF]", " ", texto)
    # Reemplazar múltiples espacios por uno solo
    texto = re.sub(r"[ \t]+", " ", texto)
    # Reducir más de 2 saltos de línea consecutivos a 2
    texto = re.sub(r"\n{3,}", "\n\n", texto)
    return texto.strip()

In [18]:

# ──  GENERAR doc_id DESDE NOMBRE DE ARCHIVO ──────────────────────
def generar_doc_id(filepath: str) -> str:
    """
    Genera un doc_id único y legible a partir del nombre del archivo.
    Ejemplo: 'biologia_celular_cap1.txt' → 'biologia_celular_cap1'
    """
    return Path(filepath).stem  # nombre sin extensión


In [20]:
# ── CARGAR Y PREPARAR DOCUMENTOS ─────────────────────────────────
def cargar_documentos(docs_dir: str) -> list[Document]:
    """
    Carga todos los archivos .txt del directorio indicado.
    Aplica limpieza y asigna metadatos base:
      - doc_id  : nombre del archivo sin extensión
      - source  : ruta completa del archivo
      - page    : 0 (TXT no tiene páginas; usamos 0 como convención)
    Retorna una lista de objetos Document de LangChain.
    """
    loader = DirectoryLoader(
        docs_dir,
        glob="**/*.txt",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"},
        show_progress=True,
    )
    docs_raw = loader.load()

    documentos_limpios = []
    for doc in docs_raw:
        texto_limpio = limpiar_texto(doc.page_content)
        doc_id = generar_doc_id(doc.metadata["source"])

        documentos_limpios.append(
            Document(
                page_content=texto_limpio,
                metadata={
                    "doc_id": doc_id,
                    "source": doc.metadata["source"],
                    "page": 0,           # TXT no tiene páginas reales
                },
            )
        )

    print(f"Documentos cargados y limpios: {len(documentos_limpios)}")
    return documentos_limpios

In [21]:

# ── SEGMENTAR EN CHUNKS ──────────────────────────────────────────
def segmentar_chunks(documentos: list[Document]) -> list[Document]:
    """
    Divide cada documento en chunks usando RecursiveCharacterTextSplitter.
    A cada chunk se le agrega metadato:
      - chunk_id : índice incremental dentro del documento (doc_id_0, doc_id_1, …)
    Retorna lista de chunks como objetos Document con metadatos completos.
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ".", " ", ""],  # Jerarquía de separadores
    )

    chunks_finales = []
    for doc in documentos:
        sub_chunks = splitter.split_documents([doc])
        for i, chunk in enumerate(sub_chunks):
            # Añadir chunk_id único: doc_id + índice
            chunk.metadata["chunk_id"] = f"{chunk.metadata['doc_id']}_{i}"
            chunks_finales.append(chunk)

    print(f"Total de chunks generados: {len(chunks_finales)}")
    return chunks_finales

In [22]:
# ── GENERAR EMBEDDINGS E INDEXAR EN FAISS ───────────────────────
def indexar_en_faiss(chunks: list[Document], faiss_dir: str) -> FAISS:
    """
    Genera embeddings con HuggingFace y los indexa en FAISS.
    Si ya existe el índice en disco, lo carga (no re-embeddea).
    Retorna el objeto vectorstore listo para hacer búsquedas.
    """
    embeddings = HuggingFaceEmbeddings(
        model_name=EMBED_MODEL,
        model_kwargs={"device": "cpu"},   # cambiar a 'cuda' si tienes GPU
        encode_kwargs={"normalize_embeddings": True},
    )

    # Si ya existe el índice, cargarlo directamente
    if Path(faiss_dir).exists():
        print(f"Índice FAISS encontrado en '{faiss_dir}'. Cargando...")
        vectorstore = FAISS.load_local(faiss_dir, embeddings, allow_dangerous_deserialization=True)
        print(f"Índice cargado con {vectorstore.index.ntotal} vectores.")
        return vectorstore

    # Crear nuevo índice
    print(f"Creando nuevo índice FAISS en '{faiss_dir}'...")
    vectorstore = FAISS.from_documents(
        documents=chunks,
        embedding=embeddings,
    )
    
    # Guardar persistente en disco
    os.makedirs(faiss_dir, exist_ok=True)
    vectorstore.save_local(faiss_dir)
    print(f"Índice creado con {vectorstore.index.ntotal} vectores.")
    return vectorstore


In [23]:
# ──  VERIFICAR METADATOS ──────────────────────
def verificar_metadatos(vectorstore: FAISS, n: int = 3):
    """
    Muestra una muestra de n chunks del índice para verificar
    que los metadatos (doc_id, page, chunk_id) están correctos.
    """
    print("\nMuestra de metadatos en el índice:")
    # Obtener documentos del vectorstore de FAISS
    docs_metadatos = list(vectorstore.docstore._dict.values())[:n]
    for i, doc in enumerate(docs_metadatos):
        print(f"\n--- Chunk {i+1} ---")
        print(f"  doc_id   : {doc.metadata.get('doc_id')}")
        print(f"  page     : {doc.metadata.get('page')}")
        print(f"  chunk_id : {doc.metadata.get('chunk_id')}")
        print(f"  texto    : {doc.page_content[:120]}...")


In [24]:
# ── PIPELINE PRINCIPAL ─────────────────────────────────────────────────────
def main():
    print("=" * 60)
    print("  RAG – Biología Celular | Ingesta e Indexación")
    print("=" * 60)

    # Paso 1: Cargar y limpiar documentos
    documentos = cargar_documentos(DOCS_DIR)

    # Paso 2: Segmentar en chunks con metadatos
    chunks = segmentar_chunks(documentos)

    # Paso 3: Embeddings + indexar en FAISS
    vectorstore = indexar_en_faiss(chunks, FAISS_DIR)

    # Paso 4: Verificar que los metadatos quedaron bien
    verificar_metadatos(vectorstore)

    print("\n🎉 Ingesta e indexación completada.")
    print(f"   Índice guardado en: {FAISS_DIR}/")
    return vectorstore


In [25]:
if __name__ == "__main__":
    vectorstore = main()

  RAG – Biología Celular | Ingesta e Indexación


100%|██████████| 65/65 [00:00<00:00, 3708.95it/s]

Documentos cargados y limpios: 65


Total de chunks generados: 12498
Creando nuevo índice FAISS en 'faiss_index'...
Índice creado con 12498 vectores.

Muestra de metadatos en el índice:

--- Chunk 1 ---
  doc_id   : Adaptation_of_the_Pivotal-Differential_Genome_Pattern_for_the_Induction_of_Intergenomic_Chromosome_Recombination_in_Hybrids_of_Synthetic_Amphidiploids_28791037
  page     : 0
  chunk_id : Adaptation_of_the_Pivotal-Differential_Genome_Pattern_for_the_Induction_of_Intergenomic_Chromosome_Recombination_in_Hybrids_of_Synthetic_Amphidiploids_28791037_0
  texto    : TITLE: Adaptation of the Pivotal-Differential Genome Pattern for the Induction of Intergenomic Chromosome Recombination ...

--- Chunk 2 ---
  doc_id   : Adaptation_of_the_Pivotal-Differential_Genome_Pattern_for_the_Induction_of_Intergenomic_Chromosome_Recombination_in_Hybrids_of_Synthetic_Amphidiploids_28791037
  page     : 0
  chunk_id : Adaptation_of_the_Pivotal-Differential_Genome_Pattern_for_the_Induction_of_Intergenomic_Chromosome_Recombination_in